# Exploratory Data Analysis — NYC Airbnb Short-Term Rental Prices

This notebook explores the raw Airbnb dataset to understand its structure, identify quality issues, and determine appropriate cleaning steps for the ML pipeline.

Key goals:
1. Understand the distribution of features, especially `price`
2. Identify missing values and data type issues
3. Determine price outlier thresholds
4. Fix data issues that will be replicated in the `basic_cleaning` pipeline step

## 1. Setup and Data Fetch

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ydata_profiling

%matplotlib inline

In [ ]:
# Initialize a W&B run for EDA — save_code=True versions this notebook in W&B
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)

# Fetch the latest sample.csv artifact from W&B
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Automated Profile Report

We use `ydata_profiling` (the modern successor to `pandas-profiling`) to generate a comprehensive EDA report.

In [ ]:
profile = ydata_profiling.ProfileReport(df)
profile.to_widgets()

### Key observations from the profile report:

- **Missing values**: `last_review` and `reviews_per_month` have significant missing values
- **Data type issue**: `last_review` is stored as a string, but should be `datetime`
- **Price outliers**: The `price` column has extreme values including zeros and very high prices
- **Geographic data**: `latitude` and `longitude` are present and can be used to filter for NYC boundaries

After discussion with stakeholders, we will restrict prices to **$10 – $350 per night**.

## 3. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price distribution
axes[0].hist(df['price'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('Price ($/night)')
axes[0].set_ylabel('Count')

# Log-scaled for better visibility
axes[1].hist(df['price'].clip(upper=500), bins=100, color='coral', edgecolor='white')
axes[1].axvline(10, color='green', linestyle='--', label='min_price=$10')
axes[1].axvline(350, color='red', linestyle='--', label='max_price=$350')
axes[1].set_title('Price Distribution (Clipped at $500)')
axes[1].set_xlabel('Price ($/night)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Prices below $10: {(df['price'] < 10).sum()}")
print(f"Prices above $350: {(df['price'] > 350).sum()}")
print(f"Price stats:\n{df['price'].describe()}")

## 4. Geographic Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    df['longitude'], df['latitude'],
    c=df['price'].clip(upper=350), cmap='viridis',
    alpha=0.3, s=2
)
plt.colorbar(scatter, ax=ax, label='Price ($/night)')
ax.set_title('NYC Airbnb Listings — Geographic Distribution by Price')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

# Check for out-of-NYC listings
out_of_bounds = ~(df['longitude'].between(-74.25, -73.50) & df['latitude'].between(40.5, 41.2))
print(f"Listings outside NYC boundaries: {out_of_bounds.sum()}")

## 5. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

**Note**: We are **not** imputing missing values in EDA. Missing value imputation will be handled inside the inference pipeline (`train_random_forest` step) so that the system can manage them at prediction time without data leakage.

## 6. Neighbourhood Group Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count distribution
df['neighbourhood_group'].value_counts().plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Listings per Neighbourhood Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Median price per neighbourhood
df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='salmon'
)
axes[1].set_title('Median Price per Neighbourhood Group')
axes[1].set_ylabel('Median Price ($/night)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Room Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['room_type'].value_counts().plot(kind='bar', ax=axes[0], color='purple', alpha=0.7)
axes[0].set_title('Listings by Room Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df.groupby('room_type')['price'].median().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='orange', alpha=0.7
)
axes[1].set_title('Median Price by Room Type')
axes[1].set_ylabel('Median Price ($/night)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Apply Basic Fixes

Based on our EDA findings, we apply two critical fixes to the data:
1. **Remove price outliers** (outside [$10, $350])
2. **Convert `last_review` to datetime**

In [ ]:
min_price = 10
max_price = 350

print(f"Shape before cleaning: {df.shape}")

# Drop price outliers
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

print(f"Shape after cleaning: {df.shape}")

## 9. Verify Fixes

In [ ]:
df.info()
print(f"\nPrice range: [{df['price'].min()}, {df['price'].max()}]")
print(f"last_review dtype: {df['last_review'].dtype}")
print(f"All prices in valid range: {df['price'].between(min_price, max_price).all()}")

## 10. EDA Summary

**Findings:**
- Dataset contains ~20,000 NYC Airbnb listings
- `last_review` was stored as string → converted to datetime
- `price` had extreme outliers (zeros and very high values) → filtered to [$10, $350]
- `reviews_per_month` and `last_review` have missing values → will be handled by the inference pipeline
- Manhattan commands the highest median prices, followed by Brooklyn
- Entire home/apt listings are significantly more expensive than private or shared rooms

**Pipeline decisions:**
- Price bounds: min=$10, max=$350 (encoded in `config.yaml`)
- Geographic filter: longitude ∈ [-74.25, -73.50], latitude ∈ [40.5, 41.2]
- Missing values will be imputed inside the sklearn pipeline (zero/constant imputation)

In [ ]:
# End the W&B run
run.finish()